In [ ]:
import torch
import torch.nn as nn

from protlig.testing_helpers import mol_to_graph, load_dataset

In [14]:
data = load_dataset()
smiles = data['SMILES']
ys = data['label']
smiles = smiles[0]
y = ys[0]

In [ ]:
import torch
import torch.nn as nn


class GCNLayerSparse(nn.Module):
    def __init__(self, in_channels, out_channels, bias=True, dtype=None):
        super().__init__()
        self.out_channels = out_channels
        self.W = nn.Parameter(torch.empty(in_channels, out_channels, dtype=dtype))
        if bias:
            self.b = nn.Parameter(torch.empty(out_channels, dtype=dtype))
        else:
            self.register_parameter("b", None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.W)
        if self.b is not None:
            nn.init.zeros_(self.b)

    def forward(self, x, edge_index):
        N = x.size(0)

        # 1. Remove existing self loops, then add exactly one per node.
        edge_index = edge_index[:, edge_index[0] != edge_index[1]]
        loops = torch.arange(N, device=edge_index.device).unsqueeze(0).repeat(2, 1)
        edge_index = torch.cat([edge_index, loops], dim=1)
        src, dst = edge_index

        # 2. In-degree at the receiving node, including the self loop (deg >= 1).
        deg = torch.zeros(N, dtype=x.dtype, device=x.device)
        deg.index_add_(0, dst, torch.ones_like(dst, dtype=x.dtype))
        d_inv_sqrt = deg.pow(-0.5)

        # 3. Symmetric normalization coefficient for each edge.
        coef = d_inv_sqrt[src] * d_inv_sqrt[dst]

        # 4. Transform, build messages, and sum them at the receiving node.
        h = x @ self.W
        msg = h[src] * coef.unsqueeze(-1)
        out = torch.zeros(N, self.out_channels, dtype=h.dtype, device=h.device)
        out.index_add_(0, dst, msg)

        if self.b is not None:
            out = out + self.b
        return out


class GCNLayerDense(nn.Module):
    def __init__(self, in_channels, out_channels, bias=True, dtype=None):
        super().__init__()
        self.W = nn.Parameter(torch.empty(in_channels, out_channels, dtype=dtype))
        if bias:
            self.b = nn.Parameter(torch.empty(out_channels, dtype=dtype))
        else:
            self.register_parameter("b", None)
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.W)
        if self.b is not None:
            nn.init.zeros_(self.b)

    def forward(self, x, edge_index):
        N = x.size(0)
        src, dst = edge_index

        # Row = receiver, column = sender (same direction as the sparse layer).
        A = torch.zeros(N, N, dtype=x.dtype, device=x.device)
        A[dst, src] = 1.0
        A.fill_diagonal_(1.0)  # exactly one self loop per node -> A_tilde

        deg = A.sum(dim=1)  # in-degree including self loop
        d_inv_sqrt = deg.pow(-0.5)
        S = d_inv_sqrt.unsqueeze(1) * A * d_inv_sqrt.unsqueeze(0)

        out = S @ (x @ self.W)
        if self.b is not None:
            out = out + self.b
        return out

tensor([[-0.9156, -0.7821, -0.4804, -0.5722, -0.5386, -0.8644, -0.8766,  0.7651,
          0.1902,  0.6727, -1.4880,  0.7771],
        [-1.7906, -1.5762, -0.7508, -0.7648, -1.0652, -1.2236, -1.5328,  1.1648,
          0.3260,  1.2873, -2.2845,  1.1808],
        [-2.8091, -2.5182, -1.1039, -1.0584, -1.7183, -1.6950, -2.2809,  1.5846,
          0.4617,  2.0485, -3.2031,  1.7072],
        [-2.2544, -2.0355, -0.9870, -1.0360, -1.4350, -1.5504, -1.8508,  1.2847,
          0.3489,  1.7177, -2.7083,  1.5337],
        [-2.4031, -2.1703, -1.2474, -1.4671, -1.5942, -2.0619, -2.0662,  1.5098,
          0.3625,  1.9325, -3.2912,  1.9637],
        [-2.0554, -1.8365, -0.8694, -0.8896, -1.2680, -1.3789, -1.7102,  1.2292,
          0.3420,  1.5219, -2.5020,  1.3558],
        [-2.0554, -1.8365, -0.8694, -0.8896, -1.2680, -1.3789, -1.7102,  1.2292,
          0.3420,  1.5219, -2.5020,  1.3558],
        [-2.8091, -2.5182, -1.1039, -1.0584, -1.7183, -1.6950, -2.2809,  1.5846,
          0.4617,  2.0485, -3.

tensor([1., 2., 3., 2., 3., 2., 2., 3., 2., 3., 2., 3., 2., 1., 3., 2., 2., 2.,
        2., 2., 3., 1., 3., 1., 3., 1., 3., 1., 3., 1., 3., 1.],
       dtype=torch.float64)